# Performance & Risk

How the portfolio has *moved* and how much *risk* sits behind that motion. Everything
here uses **today's holdings held flat** across the window (a current-composition lens —
it doesn't replay your actual trade timing), valuing each holding by its own price
history where one exists and a proxy index for proprietary funds.

*Charts are interactive: hover for values, drag to zoom, use the range sliders.*

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from invest import config, analysis as A, ledger

px.defaults.template = "plotly_white"
PALETTE = px.colors.qualitative.Safe

positions = pd.read_parquet(config.POSITIONS_PARQUET)
history = pd.read_parquet(config.PRICES_PARQUET)
transactions = pd.read_parquet(config.TRANSACTIONS_PARQUET)
TOTAL = positions["market_value"].sum()
print(f"${TOTAL:,.0f} across {positions['account_name'].nunique()} accounts "
      f"· {len(history)} trading days {history.index.min():%Y-%m-%d}→{history.index.max():%Y-%m-%d}")

## Growth of $1 — you vs the benchmarks

Cumulative return of the portfolio against the usual yardsticks (S&P 500, Nasdaq-100,
total US market). Same starting dollar; the spread at the right edge is your relative
performance.

In [ ]:
port_ret = A.portfolio_return_series(positions, history)
growth = pd.DataFrame({"Portfolio": (1 + port_ret).cumprod()})
bench = A.growth_of_dollar(history, ["SPY", "QQQ", "VTI"]).rename(
    columns={"SPY": "S&P 500", "QQQ": "Nasdaq-100", "VTI": "Total US"})
growth = growth.join(bench, how="inner")

fig = go.Figure()
for i, col in enumerate(growth.columns):
    fig.add_scatter(x=growth.index, y=growth[col], name=col, mode="lines",
                    line=dict(width=3 if col == "Portfolio" else 1.5,
                              color="#111" if col == "Portfolio" else PALETTE[i % len(PALETTE)]))
fig.update_layout(title="Growth of $1 — portfolio vs benchmarks", height=480,
                  yaxis_tickprefix="$", hovermode="x unified",
                  xaxis=dict(rangeslider=dict(visible=True), type="date"))
tot_ret = growth["Portfolio"].iloc[-1] - 1
print(f"Portfolio total return over window: {tot_ret:.1%}  (vs S&P {growth['S&P 500'].iloc[-1]-1:.1%})")
fig.show()

## Portfolio value over time, by asset class

Stacked to the current total. A composition lens — cash and proprietary pools sit flat,
index-tracking funds follow their proxy's *shape*.

In [ ]:
pvh = A.portfolio_value_history(positions, history)        # date x asset_class
pvh = pvh[pvh.iloc[-1].sort_values(ascending=False).index]   # largest sleeves at the bottom
fig = go.Figure()
for i, col in enumerate(pvh.columns):
    fig.add_scatter(x=pvh.index, y=pvh[col], name=col, mode="lines",
                    stackgroup="one", line=dict(width=0, color=PALETTE[i % len(PALETTE)]))
fig.update_layout(title="Portfolio value by asset class (current holdings)", height=480,
                  yaxis_tickprefix="$", hovermode="x unified")
fig.show()

## Underwater plot — drawdowns

Every dip below the prior peak, as a percentage. The depth and *duration* of the troughs
are the felt experience of holding this portfolio.

In [ ]:
value = pvh.sum(axis=1)
dd = A.drawdown(value)
fig = go.Figure(go.Scatter(x=dd.index, y=dd, fill="tozeroy", mode="lines",
                           line=dict(color="#c0392b"), name="drawdown"))
fig.update_layout(title=f"Underwater plot — worst drawdown {dd.min():.1%}", height=400,
                  yaxis_tickformat=".0%", hovermode="x",
                  xaxis=dict(rangeslider=dict(visible=True), type="date"))
fig.show()

## Rolling risk — volatility & beta

Left: 63-day (≈one quarter) annualized volatility of the portfolio vs the S&P. Right:
the portfolio's rolling **beta** to the market — how much of the market's move it tends
to capture (1.0 = moves one-for-one).

In [ ]:
spy_ret = history["SPY"].pct_change()
vol_p = A.rolling_volatility(port_ret, window=63)
vol_m = A.rolling_volatility(spy_ret, window=63)
beta = A.rolling_beta(port_ret, spy_ret, window=63)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Annualized volatility (63d)", "Beta to S&P 500 (63d)"))
fig.add_scatter(x=vol_p.index, y=vol_p, name="Portfolio", line=dict(color="#111"), row=1, col=1)
fig.add_scatter(x=vol_m.index, y=vol_m, name="S&P 500", line=dict(color=PALETTE[0]), row=1, col=1)
fig.add_scatter(x=beta.index, y=beta, name="beta", line=dict(color=PALETTE[2]), row=1, col=2, showlegend=False)
fig.add_hline(y=1.0, line_dash="dot", line_color="grey", row=1, col=2)
fig.update_yaxes(tickformat=".0%", row=1, col=1)
fig.update_layout(height=420, title="Rolling risk", hovermode="x unified")
print(f"Latest: portfolio vol {vol_p.iloc[-1]:.1%} vs S&P {vol_m.iloc[-1]:.1%} · beta {beta.iloc[-1]:.2f}")
fig.show()

## Risk vs reward — every holding

Annualized return against annualized volatility; **bubble size = position value**, color =
asset class. Up-and-to-the-left is the sweet spot. The dashed line is the portfolio's own
risk/return for reference.

In [ ]:
ret = A.holding_returns(positions, history)
stats = pd.DataFrame({
    "ann_return": ret.mean() * A.TRADING_DAYS,
    "ann_vol": ret.std() * np.sqrt(A.TRADING_DAYS),
})
meta = (positions[~positions["is_cash"]]
        .groupby("symbol")
        .agg(market_value=("market_value", "sum"),
             asset_class=("asset_class", "first"),
             description=("description", "first")))
scat = stats.join(meta, how="inner").rename_axis("symbol").reset_index()

fig = px.scatter(scat, x="ann_vol", y="ann_return", size="market_value", color="asset_class",
                 hover_name="symbol", hover_data={"description": True, "market_value": ":$,.0f",
                 "ann_vol": ":.1%", "ann_return": ":.1%"}, size_max=60,
                 color_discrete_sequence=PALETTE, title="Risk vs reward by holding")
p_ret, p_vol = port_ret.mean() * A.TRADING_DAYS, port_ret.std() * np.sqrt(A.TRADING_DAYS)
fig.add_hline(y=p_ret, line_dash="dot", line_color="grey")
fig.add_vline(x=p_vol, line_dash="dot", line_color="grey",
              annotation_text="portfolio", annotation_position="top")
fig.update_layout(height=520, xaxis_tickformat=".0%", yaxis_tickformat=".0%",
                  xaxis_title="annualized volatility", yaxis_title="annualized return")
fig.show()

## Risk summary table

Annualized return, volatility, max drawdown and a naive Sharpe (no risk-free subtracted)
for each priced holding and the benchmarks.

In [ ]:
summary = A.risk_summary(history)
summary.sort_values("sharpe_naive", ascending=False).style.format({
    "annual_return": "{:.1%}", "annual_vol": "{:.1%}",
    "max_drawdown": "{:.1%}", "sharpe_naive": "{:.2f}"}).background_gradient(
    subset=["sharpe_naive"], cmap="Greens")

---
*Composition lens, not a trade-by-trade performance attribution: it holds current share
counts fixed and prices the un-quotable sleeves via proxies. Prices are research-grade
(yfinance). For realized results see **income-dividends-projections.ipynb**.*